# API URL functions 0.1

Created: 2026-06-20

Develop functions for editing a template API url to add, for example, filters.

## THE FUNCTIONS

### insert_appid

Insert the user's APPId into the API call.

In [1]:
import os
from dotenv import load_dotenv
import urllib.parse as p
from estatjp import exceptions as xs

def insert_appid_dev(url):
    try:
        load_dotenv()
    except (FileNotFoundError,IOError) as e:
        e.add_note('Environment variable file (.env) not found. See README.')
        raise Exception(e)
    
    try:
        app_id = os.environ['ESTAT_APP_ID']
    except KeyError as e:
        e.add_note('Environment variable ESTAT_APP_ID not found. See README.')
        raise xs.AppIDMissingError(e)

    if app_id == None:
        raise xs.AppIDMissingError("Value of environment variable 'ESTAT_APP_ID' not found. See README.")
    
    split = p.urlsplit(url)
    qs = p.parse_qs(split.query, keep_blank_values=True)
    qs['appId'] = [app_id]
    qnew = p.urlencode(qs,doseq=True)
    splitnew = split._replace(query=qnew)
    urlrev = splitnew.geturl()
    return urlrev


### insert_filter(url, filter=dict)

This is an API url that fetches municipal codes in English from <https://www.e-stat.go.jp/en/dbview?sid=0000020101>.

`http://api.e-stat.go.jp/rest/3.0/app/getSimpleStatsData?cdCat01=A1101&cdTime=&appId=&lang=E&statsDataId=0000020101&metaGetFlg=Y&cntGetFlg=N&explanationGetFlg=Y&annotationGetFlg=Y&sectionHeaderFlg=1&replaceSpChars=0`

Our interest is in the filter key `cdTime`. In this example, at this writing, `cdTime` can have the values 1980100000, 1985100000, 1990100000, 1995100000, 2000100000, 2005100000, 2010100000, 2015100000, or 2020100000, corresponding to census years. If `cdTime` is omitted from the url, the call returns values for all census years, which requires more cells than the 100,000-cell restriction. One solution is to make multiple API calls and concatenate the results. 

The function `insert_filter_key_dev(url, key, values)` enables dynamic insertion of a key and a value or a list of values.

In [6]:
import re

def insert_filter_dev(url, key, values):
    split = p.urlsplit(url)
    qs = p.parse_qs(split.query, keep_blank_values=True)
    vallist = list(map(str, values))
    valq = ",".join(vallist)
    qs[key] = [valq]
    qnew = p.urlencode(qs,doseq=True)
    splitnew = split._replace(query=qnew)
    urlrev = splitnew.geturl()
    return urlrev



## get_csv_data_dev(url, key=None, values=None, description = datetime.datetime.now())

Modify get_csv_data to insert filters.

Also add a test for whether the query may have retrieved more than 100,000 cells and thus has been truncated.

In [3]:
import pandas as pd
import numpy
import requests
import tempfile
import re
import datetime

def get_csv_data_call(url, description = datetime.datetime.now()):
    result = {}
    try:
        with requests.get(url,stream=False) as estatresponse: # chunking in iter_lines doesn't work for stream=True
            estatresponse.raise_for_status()

            if estatresponse.encoding is None:
                estatresponse.encoding = 'utf-8'
            estatlines = estatresponse.iter_lines(chunk_size=1024, decode_unicode=True)
            with tempfile.NamedTemporaryFile(mode='w',delete_on_close=False,encoding = 'utf-8') as fheader:
                with tempfile.NamedTemporaryFile(mode='w',delete_on_close=False,encoding = 'utf-8') as fp:
                    inheader = True
                    colnum = 0
                    for line in estatlines:
                        if inheader == True:
                            #count columns
                            fields = re.split('","',line)
                            if len(fields) > colnum :
                                colnum = len(fields)
                            fheader.write(line)
                            fheader.write("\n")
                            if( line.startswith('"VALUE"')):
                                inheader = False
                                fheader.flush()
                                fheader.seek(0)
                        else:
                            fp.write(line)
                            fp.write("\n")
                    fheader.close()
                    fp.close()
                    if inheader == True:
                        errmsg = "The stream that e-Stat returned lacks a 'VALUE' line. See temp file: " + fheader.name
                        raise Exception(errmsg)
                    dfHeader = pd.read_csv(fheader.name, names = range(colnum), dtype=str)
                    dfHeader = dfHeader.dropna(axis=1, how = "all")
                    dfMain = pd.read_csv(fp.name, dtype=str)
                    result['Description'] = description
                    result['Header'] = dfHeader
                    result['Main'] = dfMain

    except requests.RequestException as e:
            raise

    return result

def get_csv_data_dev(url, key=None, values=None, description = datetime.datetime.now()):
    try:
        url = insert_appid_dev(url=url)
    except xs.AppIDError as e:
        raise
    except xs.AppIDMissingError as e:
        raise
    except Exception as e:
        e.add_note('Unhandled exception in insert_appid')
        raise

    if key != None:
        try:
            url = insert_filter_dev(url=url, key=key, values=values)
        except xs.AppIDError as e:
            raise
        except xs.AppIDMissingError as e:
            raise
        except Exception as e:
            e.add_note('Unhandled exception in insert_filter')
            raise

    # the csv has several rows of metadata terminated by a row starting with "VALUE".
    # The data table starts on the next row.
    # Put the metadata in a StringIO
    result = get_csv_data_call(url = url, description=description)
    
    # Check for truncation of cells beyond 100,000-cell limit
    dfh = result.get('Header')
    filtered_dfh = dfh.query("@dfh[0] == 'NEXT_KEY'")
    
    while filtered_dfh.empty == False:
        nextkey = filtered_dfh[1].astype(str)
        loopurl = insert_filter_dev(url, key='startPosition', values=nextkey)
        loopres = get_csv_data_call(loopurl)
        loopdfh = loopres.get('Header')
#        print(loopdfh)
#        filtered_dfh = None
        filtered_dfh = loopdfh.query("@loopdfh[0] == 'NEXT_KEY'")
        main1 = result['Main']
        mainloop = loopres['Main']
        main1 = pd.concat([main1, mainloop], ignore_index=True)
        result['Main']  =main1



    return result

## Driver code

This code block inserts a ``timekey`` and a list of census years into the API url call. Intentionally, the query retrieves more than the 100,000-cell limit and throws a TruncatedResultError. 

In [7]:
# Time codes for filtering census years
# For the municipal codes in English from https://www.e-stat.go.jp/en/dbview?sid=0000020101

municodeapiurlen = "http://api.e-stat.go.jp/rest/3.0/app/getSimpleStatsData?cdCat01=A1101&appId=&lang=E&statsDataId=0000020101&metaGetFlg=Y&cntGetFlg=N&explanationGetFlg=Y&annotationGetFlg=Y&sectionHeaderFlg=1&replaceSpChars=0"

timekey = "cdTime"
censusyears =[ 1980100000,1985100000,1990100000,1995100000,2000100000,2005100000,2010100000,2015100000,2020100000]

try:        
    dfs = get_csv_data_dev(url=municodeapiurlen, key=timekey, values=censusyears, description=datetime.datetime.now())
    print(dfs.get('Description'))
    print(dfs.get('Header'))
    dfn = dfs.get('Header')
    val = dfn.query("@dfn[0] == 'OVERALL_TOTAL_NUMBER'")[1].to_numpy(dtype='int64')[0]
    print(val)
    print(dfs.get('Main'))
except xs.TruncatedResultError as e:
    print(type(e))
    print(e.user_err_msg)
    print(e.args)
except Exception as e:
    print(type(e))
    print(e.args)
    print(e.with_traceback)

2026-06-30 10:30:23.466481
                       0                                                  1  \
0                 RESULT                                                NaN   
1                 STATUS                                                  0   
2              ERROR_MSG       The process has been successfully completed.   
3                   DATE                      2026-06-30T10:31:54.559+09:00   
4             RESULT_INF                                                NaN   
5           TOTAL_NUMBER                                              25063   
6            FROM_NUMBER                                                  1   
7              TO_NUMBER                                              25063   
8              TABLE_INF                                         0000020101   
9              STAT_NAME                                           00200502   
10               GOV_ORG                                              00200   
11       STATISTICS_NAME 

The following code block is from the first version of ``get_csv_data()`` and is included to test backward compatibility.

In [8]:
# Labor force url:
enurl = 'http://api.e-stat.go.jp/rest/3.0/app/getSimpleStatsData?appId=&lang=E&statsDataId=0003005798&metaGetFlg=Y&cntGetFlg=N&explanationGetFlg=Y&annotationGetFlg=Y&sectionHeaderFlg=1&replaceSpChars=0'

try:        
    dfs = get_csv_data_dev(enurl,description=datetime.datetime.now())
    print(dfs.get('Description'))
    print(dfs.get('Header'))
    print(dfs.get('Main'))
except Exception as e:
    print(type(e))
    print(e.args)
    print(e.with_traceback)

2026-06-30 10:33:42.639497
                       0                                                  1  \
0                 RESULT                                                NaN   
1                 STATUS                                                  0   
2              ERROR_MSG       The process has been successfully completed.   
3                   DATE                      2026-06-30T10:33:43.298+09:00   
4             RESULT_INF                                                NaN   
5           TOTAL_NUMBER                                               4755   
6            FROM_NUMBER                                                  1   
7              TO_NUMBER                                               4755   
8              TABLE_INF                                         0003005798   
9              STAT_NAME                                           00200531   
10               GOV_ORG                                              00200   
11       STATISTICS_NAME 

In [9]:
# url with problem
# this is supposed to return 9 rows and 11 columns
yearsapiurlen = "http://api.e-stat.go.jp/rest/3.0/app/getSimpleStatsData?cdCat01=A1101&cdArea=13100&appId=&lang=E&statsDataId=0000020101&metaGetFlg=Y&cntGetFlg=N&explanationGetFlg=Y&annotationGetFlg=Y&sectionHeaderFlg=1&replaceSpChars=0"

try:        
    dfs = get_csv_data_dev(yearsapiurlen,description=datetime.datetime.now())
    print(dfs.get('Description'))
    print(dfs.get('Header'))
    print(dfs.get('Main'))
except Exception as e:
    print(type(e))
    print(e.args)
    print(e.with_traceback)

2026-06-30 10:33:52.751044
                       0                                                  1  \
0                 RESULT                                                NaN   
1                 STATUS                                                  0   
2              ERROR_MSG       The process has been successfully completed.   
3                   DATE                      2026-06-30T10:33:52.996+09:00   
4             RESULT_INF                                                NaN   
5           TOTAL_NUMBER                                                  9   
6            FROM_NUMBER                                                  1   
7              TO_NUMBER                                                  9   
8              TABLE_INF                                         0000020101   
9              STAT_NAME                                           00200502   
10               GOV_ORG                                              00200   
11       STATISTICS_NAME 

## Trials relating to maximum download limits

The eStat API only allows for 100,000 rows in a single request. When a request exceeds 100,000 rows, the metadata contains a ```NEXT_KEY``` value that can be used to request the next set of rows. That value is inserted into the API url's query string with a ```startPosition``` key. The following code was used to develop the functionality but requires too much time to run everytime this notebook is executed.

<pre><code>
# The API url accesses a table with 23,022,540 cells
yearsapiurlen = "http://api.e-stat.go.jp/rest/3.0/app/getSimpleStatsData?appId=&lang=E&statsDataId=0000020101&metaGetFlg=Y&cntGetFlg=N&explanationGetFlg=Y&annotationGetFlg=Y&sectionHeaderFlg=1&replaceSpChars=0"

# yearsapiurlen = "http://api.e-stat.go.jp/rest/3.0/app/getSimpleStatsData?startPosition=100001&appId=&lang=E&statsDataId=0000020101&metaGetFlg=Y&cntGetFlg=N&explanationGetFlg=Y&annotationGetFlg=Y&sectionHeaderFlg=1&replaceSpChars=0"

try:        
    dfs = get_csv_data_dev(yearsapiurlen,description=datetime.datetime.now())
    print(dfs.get('Description'))
    print(dfs.get('Header'))
    print(dfs.get('Main'))
except Exception as e:
    print(type(e))
    print(e.args)
    print(e.with_traceback)
</code></pre>